# 📉 Cuaderno 03 — Mínimos cuadrados

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Tema 4.3. Cuando los datos traen ruido no se interpola: se ajusta.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_03_minimos_cuadrados'
TAREA_ID = 'U4_03_Minimos_Cuadrados'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = [os.getcwd()]
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except Exception:
            guardado = {}

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    else:
        print("Actualizando tu repositorio…")
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)

# --- Motor de calificación -------------------------------------------
if ES_COLAB:
    import base64
    import zlib

    ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")
    with open(ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado")
else:
    _lib = os.path.join(REPO, "lib")
    if _lib not in sys.path:
        sys.path.insert(0, _lib)
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación (número de control)
# ---------------------------------------------------------------
# En Colab se toma tu correo institucional; en local se escribe uno
# de prueba. De ahí sale el número de control que siembra tus datos.
# ---------------------------------------------------------------
if ES_COLAB:
    import requests

    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    info = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={{"Authorization": "Bearer " + creds.token}}, timeout=20).json()
    cuenta = info.get("user", {{}})
    email = cuenta.get("emailAddress", "")
    nombre = cuenta.get("displayName", "")
else:
    email = "m16330887@hermosillo.tecnm.mx"   # <-- cambia por el tuyo
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    from IPython.display import HTML, display
    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;'
        'border-radius:8px;font-weight:bold">⛔ Usa tu correo @….tecnm.mx</p>'))
    raise RuntimeError("Necesito tu correo institucional para sembrar tus datos.")

print("Alumno :", email)
print("NC     :", extraer_nc(email))
MI_TAREA = generar_tarea(email, nombre)
print("Cuaderno:", MI_TAREA["cuaderno"],
      "·", len(MI_TAREA["preguntas"]), "preguntas")

---

## Método 1 · Suma de residuos al cuadrado

El criterio de mínimos cuadrados es minimizar

$$S_r=\sum_{i=1}^{n}e_i^2=\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i-\cdots\right)^2 \tag{17.3}$$

Se usan cuadrados para que los errores no se cancelen y el mínimo sea único.

In [ ]:
#@title ✏️ Actividad a mano · Suma de residuos al cuadrado
EJ_1 = mano_enunciado("SUMA_RESIDUOS")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Suma de residuos al cuadrado
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("SUMA_RESIDUOS", [0.0])

### 💻 Programa la función

Abre `ejercicios/minimos_cuadrados/suma_residuos_cuadrados.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Suma de residuos al cuadrado
from ejercicios.minimos_cuadrados.suma_residuos_cuadrados import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_suma_residuos")

In [ ]:
#@title 📊 Laboratorio · Suma de residuos al cuadrado
visualizar.laboratorio("SUMA_RESIDUOS", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Suma de residuos al cuadrado
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

---

## Método 2 · Regresión lineal

Al derivar (17.3) e igualar a cero salen las **ecuaciones normales**:

$$n a_0+\left(\sum x_i\right)a_1=\sum y_i \tag{17.4}$$

$$\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1=\sum x_i y_i \tag{17.5}$$

y su solución,

$$a_1=\frac{n\sum x_iy_i-\sum x_i\sum y_i}{n\sum x_i^2-\left(\sum x_i\right)^2},\qquad
a_0=\bar y-a_1\bar x \tag{17.6–17.7}$$

In [ ]:
#@title ✏️ Actividad a mano · Regresión lineal
EJ_2 = mano_enunciado("REG_LINEAL")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión lineal
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("REG_LINEAL", [0.0, 0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/minimos_cuadrados/regresion_lineal.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Regresión lineal
from ejercicios.minimos_cuadrados.regresion_lineal import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_reg_lineal")

In [ ]:
#@title 📊 Laboratorio · Regresión lineal
visualizar.laboratorio("REG_LINEAL", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · Regresión lineal
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 3 · Regresión cuadrática

Para $y=a_0+a_1x+a_2x^2$ las ecuaciones normales son un sistema $3\times3$:

$$\begin{bmatrix}n&\sum x_i&\sum x_i^2\\ \sum x_i&\sum x_i^2&\sum x_i^3\\ \sum x_i^2&\sum x_i^3&\sum x_i^4\end{bmatrix}
\begin{bmatrix}a_0\\a_1\\a_2\end{bmatrix}=\begin{bmatrix}\sum y_i\\ \sum x_iy_i\\ \sum x_i^2y_i\end{bmatrix} \tag{17.19}$$

Resolverlo es resolver un sistema lineal: aquí se ve por qué sirve el capítulo 3.

In [ ]:
#@title ✏️ Actividad a mano · Regresión cuadrática
EJ_3 = mano_enunciado("REG_CUADRATICA")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión cuadrática
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("REG_CUADRATICA", [0.0, 0.0, 0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/minimos_cuadrados/regresion_cuadratica.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Regresión cuadrática
from ejercicios.minimos_cuadrados.regresion_cuadratica import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_reg_cuadratica")

In [ ]:
#@title 📊 Laboratorio · Regresión cuadrática
visualizar.laboratorio("REG_CUADRATICA", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Regresión cuadrática
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

---

## Método 4 · Error estándar de la estimación

Mide la dispersión alrededor del modelo, no alrededor de la media:

$$s_{y/x}=\sqrt{\frac{S_r}{n-(m+1)}} \tag{17.9, 17.20}$$

Se pierden $m+1$ grados de libertad porque se estimaron $m+1$ coeficientes.

In [ ]:
#@title ✏️ Actividad a mano · Error estándar de la estimación
EJ_4 = mano_enunciado("ERROR_ESTANDAR")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error estándar de la estimación
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("ERROR_ESTANDAR", [0.0])

### 💻 Programa la función

Abre `ejercicios/minimos_cuadrados/error_estandar.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Error estándar de la estimación
from ejercicios.minimos_cuadrados.error_estandar import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_error_estandar")

In [ ]:
#@title 📊 Laboratorio · Error estándar de la estimación
visualizar.laboratorio("ERROR_ESTANDAR", EJ_4)

In [ ]:
#@title 🧪 Pregunta 4 · Error estándar de la estimación
# Cuando tu función ya funcione, contéstala aquí.
pregunta(4)

---

## Método 5 · Coeficiente de determinación

Cuánto de la incertidumbre original explica el modelo:

$$r^2=\frac{S_t-S_r}{S_t} \tag{17.10}$$

Un $r^2$ cercano a 1 **no** prueba que la forma sea la correcta: hay que
graficar.

In [ ]:
#@title ✏️ Actividad a mano · Coeficiente de determinación
EJ_5 = mano_enunciado("COEF_DETERMINACION")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Coeficiente de determinación
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("COEF_DETERMINACION", [0.0])

### 💻 Programa la función

Abre `ejercicios/minimos_cuadrados/coeficiente_determinacion.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Coeficiente de determinación
from ejercicios.minimos_cuadrados.coeficiente_determinacion import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_coef_determinacion")

In [ ]:
#@title 📊 Laboratorio · Coeficiente de determinación
visualizar.laboratorio("COEF_DETERMINACION", EJ_5)

In [ ]:
#@title 🧪 Pregunta 5 · Coeficiente de determinación
# Cuando tu función ya funcione, contéstala aquí.
pregunta(5)

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["SUMA_RESIDUOS", "REG_LINEAL", "REG_CUADRATICA", "ERROR_ESTANDAR", "COEF_DETERMINACION"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()